In [ ]:
# 以下代码为底稿，已经整理到Python文件中，方便查看修改情况（0130

In [ ]:
# 基础版本，主要用来检查可行性
# 需要保证不同分支的数据在不同的服务器上，且路径相同
# 同lmy0121.py
import paramiko
import pandas as pd
import io
from scipy.io import loadmat
import numpy as np
import xlsxwriter

### 服务器
SERVERS = {
    '174': {'host': '192.168.90.176', 'user': 'mengyao.li', 'pwd': 'test1234', 
            'path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/'},
    '175': {'host': '192.168.90.175', 'user': 'mengyao.li', 'pwd': 'test1234', 
            'path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/'}
}

### cases
CASES = [
    ['pusch_compare_mrc_big/case001', 'PUSCH A3-14 BW100MHz SCS30KHz MappingTypeA 2Rx'],
    ['pusch_compare_mrc_big/case002', 'PUSCH A4-14 BW100MHz SCS30KHz MappingTypeA 2Rx'],
    ['pusch_compare_mrc_big/case003', 'PUSCH A5-14 BW100MHz SCS30KHz MappingTypeA 2Rx'],
    ['pusch_compare_mrc_big/case004', 'PUSCH A3-28 BW100MHz SCS30KHz MappingTypeA 2Rx'],
]

### 参考点
CUSTOM_POINTS = {
    0: (-2.5, 0.7),  # Case 1
    1: (10, 0.7),  # Case 2 
    2: (12.4, 0.7),  # Case 3 
    3: (19.9, 0.7),  # Case 4 
}

def parse_mat_structure(remote_file_content):
    """提取 SNR 和 Throughput"""
    data = loadmat(io.BytesIO(remote_file_content), squeeze_me=True, struct_as_record=False)
    if 'demodResult' not in data: return None, None
    
    res = data['demodResult']
    if not isinstance(res, (list, np.ndarray)): res = [res]
        
    snrs, thps = [], []
    for item in res:
        try:
            snrs.append(float(item.snr))
            thps.append(float(item.pusch.throughtput))
        except Exception: continue
    return np.array(snrs), np.array(thps)

# 
output_file = 'C:/Users/tools/Desktop/PUSCH_Comparison_Report_Final.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
workbook = writer.book
main_sheet_name = 'Simulation_Summary'
worksheet = workbook.add_worksheet(main_sheet_name)
writer.sheets[main_sheet_name] = worksheet

### 格式定义
header_fmt = workbook.add_format({'bold': True, 'font_size': 16, 'align': 'center', 'bg_color': '#D9D9D9', 'border': 1})
case_title_fmt = workbook.add_format({'bold': True, 'bg_color': "#F3EFEFE8", 'border': 1})
border_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter'})
highlight_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter', 'bg_color': "#FFFFFF", 'bold': True})
point_title_fmt = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'bg_color': "#FFFFFF"})

### 顶部标题设置
title_fmt = workbook.add_format({
    'bold': True, 
    'font_size': 11,      # 设置字体大小
    'align': 'left',      # 设置左对齐
    'valign': 'vcenter',  # 垂直居中
    'bg_color': '#D9D9D9', 
    'border': 1,
    'text_wrap': True,    # 开启换行
    'indent': 1           # 左侧缩进 1 格
})

global_title = (
    "1. 协议版本：TS38104 V17.7.0\n"
    "2. 章节名称：8.2.1节“Requirements for PUSCH with transform precoding disabled”\n"
    "3. 仿真表格：Table8.2.1.2-1 Minimum requirements for PUSCH with 70% of maximum throughput, Type A, 5 MHz channel bandwidth, 15 kHz SCS"
)

# 设置第一行的行高
worksheet.set_row(0, 55) 

# 合并单元格等
worksheet.merge_range('A1:R1', global_title, title_fmt)

current_row = 2
all_case_charts = []

for idx, (case_path, title_str) in enumerate(CASES):
    print(f"正在读取 Case {idx+1}...")
    case_data = {}
    
    for name in ['174', '175']:
        conf = SERVERS[name]
        try:
            ssh = paramiko.SSHClient()
            ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
            ssh.connect(conf['host'], username=conf['user'], password=conf['pwd'])
            full_path = f"{conf['path']}{case_path}/sfn0_slot8_ul_result.mat"
            sftp = ssh.open_sftp()
            with sftp.open(full_path, 'rb') as f:
                snr, thp = parse_mat_structure(f.read())
            sftp.close()
            ssh.close()
            if snr is not None:
                sort_idx = np.argsort(snr)
                case_data[name] = {'snr': snr[sort_idx], 'thp': thp[sort_idx]}
        except Exception as e: print(f"  -> {name} 失败: {e}")

    if '174' in case_data and '175' in case_data:
        snr_vals = case_data['174']['snr']
        thp_174 = case_data['174']['thp']
        thp_175 = case_data['175']['thp']
        c_snr, c_thp = CUSTOM_POINTS.get(idx, (None, None))
        
        # 
        worksheet.write(current_row, 0, f"Case {idx+1}: {title_str}", case_title_fmt)
        
        # 常规表头数据
        worksheet.write(current_row + 1, 0, "commit", border_fmt)
        worksheet.write(current_row + 1, 1, "snr", border_fmt)
        for i, val in enumerate(snr_vals): 
            worksheet.write(current_row + 1, i + 2, val, border_fmt)

        worksheet.write(current_row + 2, 0, 'commit1', border_fmt)
        worksheet.write(current_row + 2, 1, "Thp1", border_fmt)
        for i, val in enumerate(thp_174): 
            worksheet.write(current_row + 2, i + 2, val, border_fmt)

        worksheet.write(current_row + 3, 0, 'commit2', border_fmt)
        worksheet.write(current_row + 3, 1, "Thp2", border_fmt)
        for i, val in enumerate(thp_175): 
            worksheet.write(current_row + 3, i + 2, val, border_fmt)

        # 
        custom_col = 2 + len(snr_vals) + 2
        if c_snr is not None:
            worksheet.set_column(custom_col, custom_col, 9)
            worksheet.write(current_row, custom_col, "Required",point_title_fmt)
            worksheet.write(current_row + 1, custom_col, c_snr, highlight_fmt)
            worksheet.merge_range(current_row + 2, custom_col, current_row + 3, custom_col, c_thp, highlight_fmt)

        # 
        chart = workbook.add_chart({'type': 'scatter', 'subtype': 'straight_with_markers'})
        
        # 
        all_snrs = snr_vals.tolist() + ([c_snr] if c_snr is not None else [])
        all_thps = thp_174.tolist() + thp_175.tolist() + ([c_thp] if c_thp is not None else [])
        x_min, x_max = min(all_snrs), max(all_snrs)
        y_min = min(all_thps)

        # 曲线 1
        chart.add_series({
            'name': 'commit1',
            'categories': [main_sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
            'values':     [main_sheet_name, current_row + 2, 2, current_row + 2, 2 + len(snr_vals) - 1],
            'line': {'color': '#4F81BD'},
        })
        # 曲线 2
        chart.add_series({
            'name': 'commmit2',
            'categories': [main_sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
            'values':     [main_sheet_name, current_row + 3, 2, current_row + 3, 2 + len(snr_vals) - 1],
            'line': {'color': '#C0504D'},
        })
        # 参考点
        if c_snr is not None:
            chart.add_series({
                'name': 'Required',
                'categories': [main_sheet_name, current_row + 1, custom_col, current_row + 1, custom_col],
                'values':     [main_sheet_name, current_row + 2, custom_col, current_row + 2, custom_col],
                'marker':     {'type': 'diamond', 'size': 6, 'fill': {'color': "#ECE031"}, 'border': {'color': 'black'}},
                'line':       {'none': True},
            })

        # 设置坐标轴格式
        chart.set_title({'name': f'{title_str}', 'name_font': {'size': 12, 'bold': True}}) #Throughput Comparison: 
        chart.set_x_axis({
            'name': 'SNR (dB)',
            'min': x_min, 'max': x_max,
            'major_gridlines': {'visible': True},
            'label_position': 'low'
        })
        chart.set_y_axis({
            'name': 'Throughput',
            'min': y_min, 'max': 1.0,
            'num_format': '0.000',
            'major_gridlines': {'visible': True},
            'label_position': 'low'
        })
        
        chart.set_size({'width': 500, 'height': 340})
        all_case_charts.append(chart)
        
        current_row += 6 

### 图表的排列形式
chart_base_row = current_row + 2
for i, chart in enumerate(all_case_charts):
    r_pos, c_pos = i // 2, i % 2
    worksheet.insert_chart(chart_base_row + (r_pos * 18), c_pos * 9, chart)

writer.close()
#print("")

In [ ]:
# 整理不同服务器的csv结果，能够直接复制粘贴进 “原版” 的报告！
# 但需要服务器端先生成csv文件 (运行lmy0113)
import pandas as pd
import paramiko
import io

# 服务器配置保持不变
SERVERS = {
    '174': {'host': '192.168.90.174', 'user': 'mengyao.li', 'pwd': 'test1234', 'path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/'},
    '175': {'host': '192.168.90.175', 'user': 'mengyao.li', 'pwd': 'test1234', 'path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/'},
    '176': {'host': '192.168.90.176', 'user': 'mengyao.li', 'pwd': 'test1234', 'path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/'},
    '177': {'host': '192.168.90.177', 'user': 'mengyao.li', 'pwd': 'test1234', 'path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/'}
}

def get_remote_df(server_key, file_name):
    config = SERVERS[server_key]
    full_path = f"{config['path']}{file_name}"
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        ssh.connect(config['host'], username=config['user'], password=config['pwd'])
        sftp = ssh.open_sftp()
        with sftp.open(full_path, 'r') as f:
            content = f.read().decode('utf-8')
            df = pd.read_csv(io.StringIO(content))
        sftp.close()
        ssh.close()
        return df
    except Exception as e:
        print(f"连接服务器 {server_key} 失败: {e}")
        return None

def process_segmented_vertical(df1, df2, id1, id2):
    
    snr_indices = df1[df1['RowType'] == 'SNR'].index.tolist()
    tp_indices = df1[df1['RowType'] == 'Throughput'].index.tolist()
    
    final_rows = []
    
    for s_idx, t_idx in zip(snr_indices, tp_indices):
        
        snr_row = df1.iloc[s_idx].to_dict()
        snr_row['RowType'] = 'SNR'
        final_rows.append(snr_row)
        
        tp1_row = df1.iloc[t_idx].to_dict()
        tp1_row['RowType'] = f'TP_S{id1}'
        final_rows.append(tp1_row)
        
        try:
            tp2_row = df2.iloc[t_idx].to_dict()
            tp2_row['RowType'] = f'TP_S{id2}'
            final_rows.append(tp2_row)
        except IndexError:
            print(f"警告：服务器 {id2} 的文件行数不足。")
            
        # 添加一个空行或分隔符
        separator = {k: '' for k in snr_row.keys()}
        final_rows.append(separator)

    return pd.DataFrame(final_rows)

def merge_by_group(id1, file1, id2, file2):
    """
    主封装函数
    """
    df1 = get_remote_df(id1, file1)
    df2 = get_remote_df(id2, file2)
    
    if df1 is not None and df2 is not None:
        result = process_segmented_vertical(df1, df2, id1, id2)
        return result
    else:
        return None

if __name__ == "__main__":
    
    res = merge_by_group('176', '0122_sedft_mrc_new.csv', '177', '0122_acc100_mrc_new.csv')
    
    if res is not None:
        cols = ['RowType'] + [c for c in res.columns if c != 'RowType']
        res = res[cols]
        
        print(res.to_string(index=False))
        res.to_csv("C:/Users/tools/Desktop/PUSCH_Comparison2.csv", index=False)

RowType      Var1      Var2      Var3      Var4      Var5      Var6      Var7      Var8      Var9     Var10     Var11     Var12     Var13
    SNR      -1.7      -1.2      -0.7      -0.2       0.3       0.8       1.3       1.8       2.3       2.8       3.3       3.8       4.3
TP_S176  0.998313  0.999188  0.999875  0.999812  0.999875  0.999812  0.999875       1.0       1.0       1.0  0.999938       1.0       1.0
TP_S177   0.99875    0.9995  0.999687  0.999938  0.999938  0.999938  0.999875  0.999938       1.0       1.0  0.999938  0.999812       1.0
                                                                                                                                         
    SNR      16.5      17.0      17.5      18.0      18.5      19.0      19.5      20.0      20.5      21.0      21.5      22.0      22.5
TP_S176       1.0       1.0       1.0       1.0       1.0       1.0       1.0       1.0  0.999938       1.0       1.0       1.0       1.0
TP_S177  0.998437  0.999125  0.998

In [ ]:
# 在基础版本上调整，支持文件路径不同的场景，方便区分
# 修改参考点的位置，统一对齐
# 修改横轴的刻度值为0.5
# version2.0
# 但其实总能想办法把它们的路径设置为相同的（0130 
import paramiko
import pandas as pd
import io
from scipy.io import loadmat
import numpy as np
import xlsxwriter

# 1. 服务器基础配置
SERVERS = {
    '174': {
        'host': '192.168.90.177', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/',
        'display_name': 'acc100'  
    },
    '175': {
        'host': '192.168.90.175', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/',
        'display_name':'sedft'  
    }
}

# 2. Cases 配置 

CASES = [
    [
        {'174': 'pusch_compare_mrc_acc100/case001', '175': 'pusch_compare_mrc_master/case001'}, 
        'PUSCH A3-14 BW100MHz SCS30KHz MappingTypeA 2Rx'
    ],
    [
        {'174': 'pusch_compare_mrc_acc100/case002', '175': 'pusch_compare_mrc_master/case002'}, 
        'PUSCH A4-14 BW100MHz SCS30KHz MappingTypeA 2Rx'
    ],
    [
        {'174': 'pusch_compare_mrc_acc100/case003', '175': 'pusch_compare_mrc_master/case003'}, 
        'PUSCH A5-14 BW100MHz SCS30KHz MappingTypeA 2Rx'
    ]
]

### 3. 参考点
CUSTOM_POINTS = {
    0: (1.3, 0.7),  # Case 1
    1: (19.5, 0.7), # Case 2 
    2: (-2.5, 0.7)  # Case 3 
}

def parse_mat_structure(remote_file_content):
    """提取 SNR 和 Throughput"""
    data = loadmat(io.BytesIO(remote_file_content), squeeze_me=True, struct_as_record=False)
    if 'demodResult' not in data: return None, None
    res = data['demodResult']
    if not isinstance(res, (list, np.ndarray)): res = [res]
    snrs, thps = [], []
    for item in res:
        try:
            snrs.append(float(item.snr))
            thps.append(float(item.pusch.throughtput))
        except Exception: continue
    return np.array(snrs), np.array(thps)

output_file = 'C:/Users/tools/Desktop/PUSCH_new.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
workbook = writer.book

# 自动同步显示名称
name_174 = SERVERS['174']['display_name']
name_175 = SERVERS['175']['display_name']

main_sheet_name = f'{name_175} 和 {name_174}'
worksheet = workbook.add_worksheet(main_sheet_name)
writer.sheets[main_sheet_name] = worksheet

### 格式定义
case_title_fmt = workbook.add_format({'bold': True, 'bg_color': "#F3EFEFE8", 'border': 1})
border_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter'})
highlight_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter', 'bg_color': "#FFFFFF", 'bold': True})
point_title_fmt = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'bg_color': "#FFFFFF"})
title_fmt = workbook.add_format({'bold': True, 'font_size': 11, 'align': 'left', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1, 'text_wrap': True, 'indent': 1})

global_title = (
    "1. 协议版本：TS38104 V17.7.0\n"
    "2. 章节名称：8.2.1节“Requirements for PUSCH with transform precoding disabled”\n"
    "3. 仿真表格：Table8.2.1.2-1"
)
worksheet.set_row(0, 55) 
worksheet.merge_range('A1:R1', global_title, title_fmt)

current_row = 2
all_case_charts = []
results_cache = []
max_snr_len = 0

# --- 第一阶段：读取数据 ---
for idx, (paths, title_str) in enumerate(CASES):
    print(f"正在读取 Case {idx+1}...")
    case_data = {}
    for sid in ['174', '175']:
        conf = SERVERS[sid]
        case_sub_path = paths.get(sid)  
        try:
            ssh = paramiko.SSHClient()
            ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
            ssh.connect(conf['host'], username=conf['user'], password=conf['pwd'])
            
            full_path = f"{conf['base_path']}{case_sub_path}/sfn0_slot8_ul_result.mat"
            
            sftp = ssh.open_sftp()
            with sftp.open(full_path, 'rb') as f:
                snr, thp = parse_mat_structure(f.read())
            sftp.close()
            ssh.close()
            
            if snr is not None:
                sort_idx = np.argsort(snr)
                case_data[sid] = {'snr': snr[sort_idx], 'thp': thp[sort_idx]}
                if len(snr) > max_snr_len: max_snr_len = len(snr)
        except Exception as e: print(f"  -> 服务器 {sid} 读取失败: {e}")
    results_cache.append(case_data)

# --- 第二阶段：写入 Excel ---
for idx, ((paths, title_str), case_data) in enumerate(zip(CASES, results_cache)):
    if '174' in case_data and '175' in case_data:
        snr_vals = case_data['174']['snr']
        thp_174 = case_data['174']['thp']
        thp_175 = case_data['175']['thp']
        c_snr, c_thp = CUSTOM_POINTS.get(idx, (None, None))
        
        worksheet.write(current_row, 0, f"Case {idx+1}: {title_str}", case_title_fmt)
        
        # 写入表格数据
        worksheet.write(current_row + 1, 0, "commit", border_fmt)
        worksheet.write(current_row + 1, 1, "snr", border_fmt)
        for i, val in enumerate(snr_vals): 
            worksheet.write(current_row + 1, i + 2, val, border_fmt)

        worksheet.write(current_row + 2, 0, name_174, border_fmt)  
        worksheet.write(current_row + 2, 1, "Thp1", border_fmt)
        for i, val in enumerate(thp_174): 
            worksheet.write(current_row + 2, i + 2, val, border_fmt)

        worksheet.write(current_row + 3, 0, name_175, border_fmt)  
        worksheet.write(current_row + 3, 1, "Thp2", border_fmt)
        for i, val in enumerate(thp_175): 
            worksheet.write(current_row + 3, i + 2, val, border_fmt)

        # 动态计算 Request 列位置：max_snr_len 保证所有 Case 的 Request 列对齐
        custom_col = 2 + max_snr_len + 1 
        if c_snr is not None:
            worksheet.set_column(custom_col, custom_col, 10)
            worksheet.write(current_row, custom_col, "Request", point_title_fmt)
            worksheet.write(current_row + 1, custom_col, c_snr, highlight_fmt)
            worksheet.merge_range(current_row + 2, custom_col, current_row + 3, custom_col, c_thp, highlight_fmt)

        # 创建图表
        chart = workbook.add_chart({'type': 'scatter', 'subtype': 'straight_with_markers'})
        all_snrs = snr_vals.tolist() + ([c_snr] if c_snr is not None else [])
        all_thps = thp_174.tolist() + thp_175.tolist() + ([c_thp] if c_thp is not None else [])
        x_min, x_max = min(all_snrs), max(all_snrs)
        y_min = min(all_thps)

        # 图例名称直接引用单元格，实现自动一致
        chart.add_series({
            'name': [main_sheet_name, current_row + 2, 0], 
            'categories': [main_sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
            'values':     [main_sheet_name, current_row + 2, 2, current_row + 2, 2 + len(snr_vals) - 1],
            'line': {'color': '#4F81BD'},
        })
        chart.add_series({
            'name': [main_sheet_name, current_row + 3, 0],
            'categories': [main_sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
            'values':     [main_sheet_name, current_row + 3, 2, current_row + 3, 2 + len(snr_vals) - 1],
            'line': {'color': '#C0504D'},
        })
        if c_snr is not None:
            chart.add_series({
                'name': 'Request',
                'categories': [main_sheet_name, current_row + 1, custom_col, current_row + 1, custom_col],
                'values':     [main_sheet_name, current_row + 2, custom_col, current_row + 2, custom_col],
                'marker':     {'type': 'diamond', 'size': 7, 'fill': {'color': "#ECE031"}},
                'line':       {'none': True},
            })

        chart.set_title({'name': f'Case {idx+1}: {title_str}', 'name_font': {'size': 12, 'bold': True}})
            
        chart.set_x_axis({
                'name': 'SNR (dB)', 
                'min': x_min, 
                'max': x_max,
                'major_unit': 0.5,
                'major_gridlines': {'visible': True},
                'crossing': -200,  
                'axis_position': 'on_tick'
            })
        chart.set_y_axis({
                'name': 'Throughput', 
                'min': y_min, 
                'max': 1.0, 
                'num_format': '0.000', 
                'major_gridlines': {'visible': True}
            })
            
        chart.set_size({'width': 500, 'height': 340})
        all_case_charts.append(chart)
            
        current_row += 6 

# 图表排版
chart_base_row = current_row + 2
for i, chart in enumerate(all_case_charts):
    r_pos, c_pos = i // 2, i % 2
    worksheet.insert_chart(chart_base_row + (r_pos * 17), c_pos * 8, chart)

writer.close()
print("Excel 生成完毕！")

正在读取 Case 1...
正在读取 Case 2...
正在读取 Case 3...
Excel 生成完毕！


In [13]:
# 修改大标题
# 修改小标题为原来的版式
# version3.0

import paramiko
import pandas as pd
import io
from scipy.io import loadmat
import numpy as np
import xlsxwriter

# 1. 服务器基础配置
SERVERS = {
    '174': {
        'host': '192.168.90.177', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/pusch_compare_3cases/',
        'display_name': 'acc100'  
    },
    '175': {
        'host': '192.168.90.176', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/pusch_compare_3cases/',
        'display_name': 'sedft'  
    }
}

# 2. Cases 配置

CASES = [
    [
        {'174': 'pusch_compare_mrc_acc100/case001', '175': 'pusch_compare_mrc/case001'}, 
        'case001 (TDLB100-400 + 2T2R + G-FR1-A3-27)'
    ],
    [
        {'174': 'pusch_compare_mrc_acc100/case002', '175': 'pusch_compare_mrc/case002'}, 
        'case002 (TDLC300-100 + 2T2R + G-FR1-A4-27)'
    ],
    [
        {'174': 'pusch_compare_mrc_acc100/case003', '175': 'pusch_compare_mrc/case003'}, 
        'case003 (TDLB100-400 + 1T2R + G-FR1-A3-32)'
    ]
]

# 3. 参考点
CUSTOM_POINTS = {
    0: (1.3, 0.7),  # Case 1
    1: (19.5, 0.7), # Case 2 
    2: (-2.5, 0.7)  # Case 3 
}

def parse_mat_structure(remote_file_content):
    """提取 SNR 和 Throughput"""
    data = loadmat(io.BytesIO(remote_file_content), squeeze_me=True, struct_as_record=False)
    if 'demodResult' not in data: return None, None
    res = data['demodResult']
    if not isinstance(res, (list, np.ndarray)): res = [res]
    snrs, thps = [], []
    for item in res:
        try:
            snrs.append(float(item.snr))
            thps.append(float(item.pusch.throughtput))
        except Exception: continue
    return np.array(snrs), np.array(thps)

output_file = 'C:/Users/tools/Desktop/PUSCH_mrc_0203.xlsx'
writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
workbook = writer.book

# 自动同步显示名称（表格和图的commit)
name_174 = SERVERS['174']['display_name']
name_175 = SERVERS['175']['display_name']

main_sheet_name = f'{name_175} 和 {name_174}'
worksheet = workbook.add_worksheet(main_sheet_name)
writer.sheets[main_sheet_name] = worksheet

### 格式定义
case_title_fmt = workbook.add_format({'bold': True, 'bg_color': "#F3EFEFE8", 'border': 1})
border_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter'})
highlight_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter', 'bg_color': "#FFFFFF", 'bold': True})
point_title_fmt = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'bg_color': "#FFFFFF"})
title_fmt = workbook.add_format({'bold': True, 'font_size': 11, 'align': 'left', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1, 'text_wrap': True, 'indent': 1})

global_title_1 = (
    "1. 协议版本：TS38104 V17.7.0\n"
    "2. 章节名称：8.2.1节“Requirements for PUSCH with transform precoding disabled”\n"
    "3. 仿真表格：Table8.2.1.2-6"
)


global_title_2 = (
    "1. 协议版本：TS38104 V17.7.0\n"
    "2. 章节名称：8.2.2节“Requirements for PUSCH with transform precoding enabled”\n"
    "3. 仿真表格：Table8.2.2.2-2"
)

# 写入第一个大标题
worksheet.set_row(0, 55) 
worksheet.merge_range('A1:R1', global_title_1, title_fmt)

current_row = 2
all_case_charts = []
results_cache = []
max_snr_len = 0

# --- 第一阶段：读取数据 ---
for idx, (paths, title_str) in enumerate(CASES):
    print(f"正在读取 Case {idx+1}...")
    case_data = {}
    for sid in ['174', '175']:
        conf = SERVERS[sid]
        case_sub_path = paths.get(sid)  
        try:
            ssh = paramiko.SSHClient()
            ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
            ssh.connect(conf['host'], username=conf['user'], password=conf['pwd'])
            
            full_path = f"{conf['base_path']}{case_sub_path}/sfn0_slot8_ul_result.mat"
            
            sftp = ssh.open_sftp()
            with sftp.open(full_path, 'rb') as f:
                snr, thp = parse_mat_structure(f.read())
            sftp.close()
            ssh.close()
            
            if snr is not None:
                sort_idx = np.argsort(snr)
                case_data[sid] = {'snr': snr[sort_idx], 'thp': thp[sort_idx]}
                if len(snr) > max_snr_len: max_snr_len = len(snr)
        except Exception as e: print(f"  -> 服务器 {sid} 读取失败: {e}")
    results_cache.append(case_data)

# --- 第二阶段：写入 Excel ---
for idx, ((paths, title_str), case_data) in enumerate(zip(CASES, results_cache)):
    if '174' in case_data and '175' in case_data:
        # 如果是 Case 3 (索引为 2)，先插入一个新的大标题
        if idx == 2:
            current_row += 1 # 留一点空行
            worksheet.set_row(current_row, 55)
            worksheet.merge_range(current_row, 0, current_row, 17, global_title_2, title_fmt)
            current_row += 2  # 指向下一行准备写 Case 标题

        snr_vals = case_data['174']['snr']
        thp_174 = case_data['174']['thp']
        thp_175 = case_data['175']['thp']
        c_snr, c_thp = CUSTOM_POINTS.get(idx, (None, None))
        
        worksheet.write(current_row, 0, f"Case {idx+1}: {title_str}", case_title_fmt)
        
        # 写入表格数据
        worksheet.write(current_row + 1, 0, "commit", border_fmt)
        worksheet.write(current_row + 1, 1, "snr", border_fmt)
        for i, val in enumerate(snr_vals): 
            worksheet.write(current_row + 1, i + 2, val, border_fmt)

        worksheet.write(current_row + 2, 0, name_174, border_fmt)  
        worksheet.write(current_row + 2, 1, "Thp1", border_fmt)
        for i, val in enumerate(thp_174): 
            worksheet.write(current_row + 2, i + 2, val, border_fmt)

        worksheet.write(current_row + 3, 0, name_175, border_fmt)  
        worksheet.write(current_row + 3, 1, "Thp2", border_fmt)
        for i, val in enumerate(thp_175): 
            worksheet.write(current_row + 3, i + 2, val, border_fmt)

        # 动态计算 Request 列位置：max_snr_len,保证所有 Case 的 Request 列对齐
        custom_col = 2 + max_snr_len + 1 
        if c_snr is not None:
            worksheet.set_column(custom_col, custom_col, 10)
            worksheet.write(current_row, custom_col, "Request", point_title_fmt)
            worksheet.write(current_row + 1, custom_col, c_snr, highlight_fmt)
            worksheet.merge_range(current_row + 2, custom_col, current_row + 3, custom_col, c_thp, highlight_fmt)

        # 创建图表
        chart = workbook.add_chart({'type': 'scatter', 'subtype': 'straight_with_markers'})
        all_snrs = snr_vals.tolist() + ([c_snr] if c_snr is not None else [])
        all_thps = thp_174.tolist() + thp_175.tolist() + ([c_thp] if c_thp is not None else [])
        x_min, x_max = min(all_snrs), max(all_snrs)
        y_min = min(all_thps)

        # 图例名称实现自动一致
        chart.add_series({
            'name': [main_sheet_name, current_row + 2, 0], 
            'categories': [main_sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
            'values':     [main_sheet_name, current_row + 2, 2, current_row + 2, 2 + len(snr_vals) - 1],
            'line': {'color': '#4F81BD'},
        })
        chart.add_series({
            'name': [main_sheet_name, current_row + 3, 0],
            'categories': [main_sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
            'values':     [main_sheet_name, current_row + 3, 2, current_row + 3, 2 + len(snr_vals) - 1],
            'line': {'color': '#C0504D'},
        })
        if c_snr is not None:
            chart.add_series({
                'name': 'Request',
                'categories': [main_sheet_name, current_row + 1, custom_col, current_row + 1, custom_col],
                'values':     [main_sheet_name, current_row + 2, custom_col, current_row + 2, custom_col],
                'marker':     {'type': 'diamond', 'size': 7, 'fill': {'color': "#ECE031"}},
                'line':       {'none': True},
            })

        chart.set_title({'name': f'Case {idx+1}: {title_str}', 'name_font': {'size': 12, 'bold': True}})
            
        chart.set_x_axis({
                'name': 'SNR (dB)', 
                'min': x_min, 
                'max': x_max,
                'major_unit': 0.5,
                'major_gridlines': {'visible': True},
                'crossing': -200,  
                'axis_position': 'on_tick'
            })
        chart.set_y_axis({
                'name': 'Throughput', 
                'min': y_min, 
                'max': 1.0, 
                'num_format': '0.000', 
                'major_gridlines': {'visible': True}
            })
            
        chart.set_size({'width': 500, 'height': 340})
        all_case_charts.append(chart)
            
        current_row += 6 

# 图表排版
chart_base_row = current_row + 2
for i, chart in enumerate(all_case_charts):
    r_pos, c_pos = i // 2, i % 2
    worksheet.insert_chart(chart_base_row + (r_pos * 17), c_pos * 8, chart)

writer.close()
#print("Excel 生成完毕！")

正在读取 Case 1...
正在读取 Case 2...
正在读取 Case 3...


In [ ]:
# 在version 3.0 的基础上，封装为函数，方便直接生成不同的比较结果（sheet
# version 4.0
import paramiko
import pandas as pd
import io
from scipy.io import loadmat
import numpy as np
import xlsxwriter

# 1. 服务器基础配置 
SERVERS = {
    '174': {
        'host': '192.168.90.174', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/',
        'display_name': 'acc100'  
    },
    '175': {
        'host': '192.168.90.175', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/',
        'display_name': 'master'  
    },
    '176': {
        'host': '192.168.90.176', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/',
        'display_name': 'dev_v1'  
    },
    '177': {
        'host': '192.168.90.177', 
        'user': 'mengyao.li', 
        'pwd': 'test1234', 
        'base_path': '/home/mengyao.li/Code/ts_nr_simulation/test_and_sim/ulsim/',
        'display_name': 'dev_v2'  
    }
}

# 2. Cases 配置
CASES = [
    [
        {'174': 'pusch_compare_mrc/case001', '175': 'pusch_compare_mrc/case001', '176': 'pusch_compare_mrc/case001', '177': 'pusch_compare_mrc/case001'}, 
        'case001 (TDLB100-400 + 2T2R + G-FR1-A3-27)'
    ],
    [
        {'174': 'pusch_compare_mrc/case002', '175': 'pusch_compare_mrc/case002', '176': 'pusch_compare_mrc/case002', '177': 'pusch_compare_mrc/case002'}, 
        'case002 (TDLC300-100 + 2T2R + G-FR1-A4-27)'
    ],
    [
        {'174': 'pusch_compare_mrc/case003', '175': 'pusch_compare_mrc/case003', '176': 'pusch_compare_mrc/case003', '177': 'pusch_compare_mrc/case003'}, 
        'case003 (TDLB100-400 + 1T2R + G-FR1-A3-32)'
    ]
]

# 3. 参考点
CUSTOM_POINTS = {
    0: (1.3, 0.7),  # Case 1
    1: (19.5, 0.7), # Case 2 
    2: (-2.5, 0.7)  # Case 3 
}

def parse_mat_structure(remote_file_content):
    """提取 SNR 和 Throughput"""
    data = loadmat(io.BytesIO(remote_file_content), squeeze_me=True, struct_as_record=False)
    if 'demodResult' not in data: return None, None
    res = data['demodResult']
    if not isinstance(res, (list, np.ndarray)): res = [res]
    snrs, thps = [], []
    for item in res:
        try:
            snrs.append(float(item.snr))
            thps.append(float(item.pusch.throughtput))
        except Exception: continue
    return np.array(snrs), np.array(thps)

def generate_comparison_sheet(writer, server_id_a, server_id_b, sheet_name=None):
    """
    封装的 Sheet 生成函数
    server_id_a:  
    server_id_b: 
    """
    workbook = writer.book
    conf_a = SERVERS[server_id_a]
    conf_b = SERVERS[server_id_b]
    
    name_a = conf_a['display_name']
    name_b = conf_b['display_name']
    
    if not sheet_name:
        sheet_name = f'{name_a} 和 {name_b}'
    
    worksheet = workbook.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = worksheet

    # --- 格式定义 (保持原版) ---
    case_title_fmt = workbook.add_format({'bold': True, 'bg_color': "#F3EFEFE8", 'border': 1})
    border_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter'})
    highlight_fmt = workbook.add_format({'border': 1, 'align': 'center', 'valign': 'vcenter', 'bg_color': "#FFFFFF", 'bold': True})
    point_title_fmt = workbook.add_format({'bold': True, 'border': 1, 'align': 'center', 'bg_color': "#FFFFFF"})
    title_fmt = workbook.add_format({'bold': True, 'font_size': 11, 'align': 'left', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1, 'text_wrap': True, 'indent': 1})

    global_title_1 = (
        "1. 协议版本：TS38104 V17.7.0\n"
        "2. 章节名称：8.2.1节“Requirements for PUSCH with transform precoding disabled”\n"
        "3. 仿真表格：Table8.2.1.2-6"
    )
    global_title_2 = (
        "1. 协议版本：TS38104 V17.7.0\n"
        "2. 章节名称：8.2.2节“Requirements for PUSCH with transform precoding enabled”\n"
        "3. 仿真表格：Table8.2.2.2-2"
    )

    worksheet.set_row(0, 55) 
    worksheet.merge_range('A1:R1', global_title_1, title_fmt)

    current_row = 2
    all_case_charts = []
    results_cache = []
    max_snr_len = 0

    # --- 第一阶段：读取数据 ---
    for idx, (paths, title_str) in enumerate(CASES):
        print(f"[{sheet_name}] 正在读取 Case {idx+1}...")
        case_data = {}
        for sid in [server_id_a, server_id_b]:
            conf = SERVERS[sid]
            case_sub_path = paths.get(sid)  
            try:
                ssh = paramiko.SSHClient()
                ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
                ssh.connect(conf['host'], username=conf['user'], password=conf['pwd'])
                
                full_path = f"{conf['base_path']}{case_sub_path}/sfn0_slot8_ul_result.mat"
                
                sftp = ssh.open_sftp()
                with sftp.open(full_path, 'rb') as f:
                    snr, thp = parse_mat_structure(f.read())
                sftp.close()
                ssh.close()
                
                if snr is not None:
                    sort_idx = np.argsort(snr)
                    case_data[sid] = {'snr': snr[sort_idx], 'thp': thp[sort_idx]}
                    if len(snr) > max_snr_len: max_snr_len = len(snr)
            except Exception as e: 
                print(f"  -> 服务器 {sid} ({conf['host']}) 读取失败: {e}")
        results_cache.append(case_data)

    # --- 第二阶段：写入 Excel ---
    for idx, ((paths, title_str), case_data) in enumerate(zip(CASES, results_cache)):
        if server_id_a in case_data and server_id_b in case_data:
            if idx == 2:
                current_row += 1 
                worksheet.set_row(current_row, 55)
                worksheet.merge_range(current_row, 0, current_row, 17, global_title_2, title_fmt)
                current_row += 2 

            snr_vals = case_data[server_id_b]['snr']
            thp_b = case_data[server_id_b]['thp'] 
            thp_a = case_data[server_id_a]['thp'] 
            c_snr, c_thp = CUSTOM_POINTS.get(idx, (None, None))
            
            worksheet.write(current_row, 0, f"Case {idx+1}: {title_str}", case_title_fmt)
            
            worksheet.write(current_row + 1, 0, "commit", border_fmt)
            worksheet.write(current_row + 1, 1, "snr", border_fmt)
            for i, val in enumerate(snr_vals): 
                worksheet.write(current_row + 1, i + 2, val, border_fmt)

            worksheet.write(current_row + 2, 0, name_b, border_fmt)  
            worksheet.write(current_row + 2, 1, "Thp1", border_fmt)
            for i, val in enumerate(thp_b): 
                worksheet.write(current_row + 2, i + 2, val, border_fmt)

            worksheet.write(current_row + 3, 0, name_a, border_fmt)  
            worksheet.write(current_row + 3, 1, "Thp2", border_fmt)
            for i, val in enumerate(thp_a): 
                worksheet.write(current_row + 3, i + 2, val, border_fmt)

            custom_col = 2 + max_snr_len + 1 
            if c_snr is not None:
                worksheet.set_column(custom_col, custom_col, 10)
                worksheet.write(current_row, custom_col, "Request", point_title_fmt)
                worksheet.write(current_row + 1, custom_col, c_snr, highlight_fmt)
                worksheet.merge_range(current_row + 2, custom_col, current_row + 3, custom_col, c_thp, highlight_fmt)

            # 图表
            chart = workbook.add_chart({'type': 'scatter', 'subtype': 'straight_with_markers'})
            all_snrs = snr_vals.tolist() + ([c_snr] if c_snr is not None else [])
            all_thps = thp_b.tolist() + thp_a.tolist() + ([c_thp] if c_thp is not None else [])
            x_min, x_max = min(all_snrs), max(all_snrs)
            y_min = min(all_thps)

            chart.add_series({
                'name': [sheet_name, current_row + 2, 0], 
                'categories': [sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
                'values':     [sheet_name, current_row + 2, 2, current_row + 2, 2 + len(snr_vals) - 1],
                'line': {'color': '#4F81BD'},
            })
            chart.add_series({
                'name': [sheet_name, current_row + 3, 0],
                'categories': [sheet_name, current_row + 1, 2, current_row + 1, 2 + len(snr_vals) - 1],
                'values':     [sheet_name, current_row + 3, 2, current_row + 3, 2 + len(snr_vals) - 1],
                'line': {'color': '#C0504D'},
            })
            if c_snr is not None:
                chart.add_series({
                    'name': 'Request',
                    'categories': [sheet_name, current_row + 1, custom_col, current_row + 1, custom_col],
                    'values':     [sheet_name, current_row + 2, custom_col, current_row + 2, custom_col],
                    'marker':     {'type': 'diamond', 'size': 7, 'fill': {'color': "#ECE031"}},
                    'line':       {'none': True},
                })

            chart.set_title({'name': f'Case {idx+1}: {title_str}', 'name_font': {'size': 12, 'bold': True}})
            chart.set_x_axis({'name': 'SNR (dB)', 'min': x_min, 'max': x_max, 'major_unit': 0.5, 'major_gridlines': {'visible': True}, 'crossing': -200, 'axis_position': 'on_tick'})
            chart.set_y_axis({'name': 'Throughput', 'min': y_min, 'max': 1.0, 'num_format': '0.000', 'major_gridlines': {'visible': True}})
            chart.set_size({'width': 500, 'height': 340})
            all_case_charts.append(chart)
            current_row += 6 

    # 图表排版
    chart_base_row = current_row + 2
    for i, chart in enumerate(all_case_charts):
        r_pos, c_pos = i // 2, i % 2
        worksheet.insert_chart(chart_base_row + (r_pos * 17), c_pos * 8, chart)

# --- 主程序执行部分 ---
output_path = 'C:/Users/tools/Desktop/PUSCH_MultiCompare.xlsx'
writer = pd.ExcelWriter(output_path, engine='xlsxwriter')

TASKS = [
    ('177', '174', 'acc100_vs_masterwinsize'),
    ('177', '175', 'acc100_vs_winsize'),
    ('177', '176', 'acc100_vs_sedft')
]

for s_a, s_b, s_name in TASKS:
    generate_comparison_sheet(writer, s_a, s_b, s_name)

writer.close()
#print(f"Excel 生成完毕！文件保存至: {output_path}")